#Section-B

##Question B1

In [39]:
corpus = """
Cats are popular pets because they are independent, clean, and relatively
easy to take care of. They can live comfortably in homes and apartments,
and they usually do not need as much space as some other pets.

Cats need a balanced diet to stay healthy. Pet cats are commonly given
commercial cat food that contains the nutrients they need. Fresh drinking
water should always be available, and the amount of food should depend on
the cat's age, size, and activity level.

Grooming is another important part of taking care of a cat. Cats clean
themselves regularly, but brushing their fur can help remove loose hair
and reduce shedding. Regular nail trimming and dental care can also help
keep a cat healthy.

Cats are playful and need regular physical and mental activity. Toys,
scratching posts, and simple games can keep them active and reduce boredom.
Spending time playing with a cat can also help build a strong bond between
the pet and its owner.

Regular veterinary care is important for pet cats. Vaccinations, routine
checkups, parasite prevention, and other treatments can help prevent health
problems. Owners should also watch for changes in eating habits, behavior,
or activity.

Cats can be loving and enjoyable companions when they are given proper
care and attention. Providing healthy food, a safe environment, regular
healthcare, and enough playtime can help a pet cat live a comfortable and
happy life.
"""

In [40]:
# Fixed-size chunking
def chunking(text, size=250, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + size
        chunk = text[start:end]
        chunks.append(chunk.strip())

        if end >= len(text):
            break

        start = end - overlap

    return chunks


chunks = chunking(corpus, size=250, overlap=50)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Number of chunks: 7

--- Chunk 1 ---
Cats are popular pets because they are independent, clean, and relatively
easy to take care of. They can live comfortably in homes and apartments,
and they usually do not need as much space as some other pets.

Cats need a balanced diet to stay heal

--- Chunk 2 ---
ther pets.

Cats need a balanced diet to stay healthy. Pet cats are commonly given
commercial cat food that contains the nutrients they need. Fresh drinking
water should always be available, and the amount of food should depend on
the cat's age, size

--- Chunk 3 ---
mount of food should depend on
the cat's age, size, and activity level.

Grooming is another important part of taking care of a cat. Cats clean
themselves regularly, but brushing their fur can help remove loose hair
and reduce shedding. Regular nail

--- Chunk 4 ---
move loose hair
and reduce shedding. Regular nail trimming and dental care can also help
keep a cat healthy.

Cats are playful and need regular physical and ment

**Chunk Size and Overlap**

I have selected a chunk size of 250 characters and overlap of 50 characters.

Chunk size = 250: Keeps each chunk small for efficient retrieval.

Overlap = 50: Preserves some information between the chunks.
Overlap is useful because an important sentence or concept may otherwise be divided between two chunks.

A larger chunk can contain more context but may retrieve unnecessary information. A smaller chunk provides more efficient retrieval but sometimes it may lose some context

##Question B2

In [41]:
!pip install -q sentence-transformers scikit-learn

In [42]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for all chunks
cembed = model.encode(chunks)

print("Embedding shape:", cembed.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (7, 384)


This model converts every chunk into a numerical vector.

In [43]:
def dense_retrieval(query, chunks, cembed, top_k=3):

    # Convert query into embedding
    qembed = model.encode([query])

    # Calculate cosine similarity
    similarities = cosine_similarity(
        qembed,
        cembed
    )[0]

    # Get indices of top-k results
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": similarities[idx]
        })

    return results

Query 1: "What food does a pet cat need?"

The food-related chunk should come at the top because it directly talks about a cat's diet and nutrients.
The other chunks may also appear, but with lower similarity scores.
This shows that dense retrieval finds content based on the meaning of the query.

Query 2: "How can I keep my cat healthy?"

The chunks about diet, grooming, exercise, and veterinary care should be in  the top results.
Since the question is not specific, more than one chunk can be retrived.
Dense retrieval is useful to find these related chunks even when the exact words don't match.

In [44]:
query1 = "What food does a pet cat need?"

results1 = dense_retrieval(
    query1,
    chunks,
    cembed,
    top_k=3
)

print("Query:", query1)

for i, result in enumerate(results1):
    print(f"\nRank {i+1}")
    print("Similarity:", round(result["score"], 4))
    print(result["chunk"])

Query: What food does a pet cat need?

Rank 1
Similarity: 0.6959
ther pets.

Cats need a balanced diet to stay healthy. Pet cats are commonly given
commercial cat food that contains the nutrients they need. Fresh drinking
water should always be available, and the amount of food should depend on
the cat's age, size

Rank 2
Similarity: 0.6183
mount of food should depend on
the cat's age, size, and activity level.

Grooming is another important part of taking care of a cat. Cats clean
themselves regularly, but brushing their fur can help remove loose hair
and reduce shedding. Regular nail

Rank 3
Similarity: 0.5874
n be loving and enjoyable companions when they are given proper
care and attention. Providing healthy food, a safe environment, regular
healthcare, and enough playtime can help a pet cat live a comfortable and
happy life.


In [45]:
query2 = "How can I keep my cat healthy?"

results2 = dense_retrieval(
    query2,
    chunks,
    cembed,
    top_k=3
)

print("Query:", query2)

for i, result in enumerate(results2):
    print(f"\nRank {i+1}")
    print("Similarity:", round(result["score"], 4))
    print(result["chunk"])

Query: How can I keep my cat healthy?

Rank 1
Similarity: 0.7143
move loose hair
and reduce shedding. Regular nail trimming and dental care can also help
keep a cat healthy.

Cats are playful and need regular physical and mental activity. Toys,
scratching posts, and simple games can keep them active and reduce bor

Rank 2
Similarity: 0.6507
n be loving and enjoyable companions when they are given proper
care and attention. Providing healthy food, a safe environment, regular
healthcare, and enough playtime can help a pet cat live a comfortable and
happy life.

Rank 3
Similarity: 0.6321
cats. Vaccinations, routine
checkups, parasite prevention, and other treatments can help prevent health
problems. Owners should also watch for changes in eating habits, behavior,
or activity.

Cats can be loving and enjoyable companions when they are


### Dense Retrieval vs BM25

| **Query** | **Dense Retrieval** | **BM25** |
| ------------------------------------------- | ------------------------------- | ----------------------------------------------- |
| What food does a pet cat need? | Finds related information about cat food and diet | Finds words like cat, food, and diet |
| How can I keep my cat healthy? | Finds information about diet, grooming, and healthcare | Finds words like cat, healthy, and health |

### Which method worked better and why?

Both methods worked, but in different ways.

Dense retrieval focuses on the meaning of the query, while **BM25** mainly matches keywords.  
Dense retrieval can find relevant information even when the exact words are not present.  
BM25 is useful when the query contains specific words that appear directly in the text.

**Conclusion:** Dense retrieval is useful for meaning-based searches, while BM25 is better when exact words matter.

##Question B3


In [46]:
!pip install -q rank_bm25

In [47]:
from rank_bm25 import BM25Okapi

# Tokenize chunks
tokenized_chunks = [
    chunk.lower().split()
    for chunk in chunks
]

# Create BM25 index
bm25 = BM25Okapi(tokenized_chunks)

In [48]:
def bm25_retrieval(query, chunks, bm25, top_k=3):

    query_tokens = query.lower().split()

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": scores[idx]
        })

    return results

In [49]:
bm25_results1 = bm25_retrieval(
    query1,
    chunks,
    bm25,
    top_k=3
)

print("Query:", query1)

for i, result in enumerate(bm25_results1):
    print(f"\nRank {i+1}")
    print("BM25 Score:", round(result["score"], 4))
    print(result["chunk"])

Query: What food does a pet cat need?

Rank 1
BM25 Score: 1.8864
ther pets.

Cats need a balanced diet to stay healthy. Pet cats are commonly given
commercial cat food that contains the nutrients they need. Fresh drinking
water should always be available, and the amount of food should depend on
the cat's age, size

Rank 2
BM25 Score: 1.0481
mount of food should depend on
the cat's age, size, and activity level.

Grooming is another important part of taking care of a cat. Cats clean
themselves regularly, but brushing their fur can help remove loose hair
and reduce shedding. Regular nail

Rank 3
BM25 Score: 1.0225
n be loving and enjoyable companions when they are given proper
care and attention. Providing healthy food, a safe environment, regular
healthcare, and enough playtime can help a pet cat live a comfortable and
happy life.


In [50]:
bm25_results2 = bm25_retrieval(
    query2,
    chunks,
    bm25,
    top_k=3
)

print("Query:", query2)

for i, result in enumerate(bm25_results2):
    print(f"\nRank {i+1}")
    print("BM25 Score:", round(result["score"], 4))
    print(result["chunk"])

Query: How can I keep my cat healthy?

Rank 1
BM25 Score: 1.7722
move loose hair
and reduce shedding. Regular nail trimming and dental care can also help
keep a cat healthy.

Cats are playful and need regular physical and mental activity. Toys,
scratching posts, and simple games can keep them active and reduce bor

Rank 2
BM25 Score: 1.4345
d simple games can keep them active and reduce boredom.
Spending time playing with a cat can also help build a strong bond between
the pet and its owner.

Regular veterinary care is important for pet cats. Vaccinations, routine
checkups, parasite pre

Rank 3
BM25 Score: 0.5766
n be loving and enjoyable companions when they are given proper
care and attention. Providing healthy food, a safe environment, regular
healthcare, and enough playtime can help a pet cat live a comfortable and
happy life.


In this part, I have used BM25 to search the same cat-related chunks used in dense retrieval. BM25 mainly matches the important words in the query with the words present in the chunks.

SAme two Queries are used:

Query 1: What food does a pet cat need?
Query 2: How can I keep my cat healthy?

Then the results were compared with dense retrieval. BM25 worked better when the query contained words that directly present in the text, on the other hand dense retrieval was better at finding related information based on meaning.

Why: BM25 focuses on keyword matching, whereas dense retrieval focuses more on the meaning of the query.